# 最终参数搜索

- 对应论文章节：第3章 RAG算法实验分析
- 源脚本：`experiments/02_消融实验/scripts/运行_最终参数搜索.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准


对应 4/6、5/7 等参数比较

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/02_消融实验/scripts/运行_最终参数搜索.py
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""运行论文最终参数配置（4/6、5/7）的对比搜索。"""

from __future__ import annotations

import argparse
import json
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.common import DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, ensure_dir
from retrieval_pipeline.datasets import load_crud_cases
from retrieval_pipeline.metrics import evaluate_crud_results
from retrieval_pipeline.pipeline import PipelineVariant, RagExperimentPipeline

OUTPUT_ROOT = ROOT / "results" / "02_最终参数搜索_20260517"
CRUD_SUBSET_SPLITS = {"questanswer_2docs": 797, "questanswer_3docs": 797}

VARIANTS = (
    PipelineVariant(
        key="base_router_4_6",
        label="当前最佳 4/6 + 按题作答",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="task_router",
        multi_snippet_count=1,
        final_source_count=4,
        complex_source_count=6,
        selection_mode="aspect_cover_v2",
    ),
    PipelineVariant(
        key="task_aligned_4_6",
        label="当前最佳 4/6 + 强制对齐",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="task_aligned",
        multi_snippet_count=1,
        final_source_count=4,
        complex_source_count=6,
        selection_mode="aspect_cover_v2",
    ),
    PipelineVariant(
        key="router_5_7",
        label="当前最佳 5/7 + 按题作答",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="task_router",
        multi_snippet_count=1,
        final_source_count=5,
        complex_source_count=7,
        selection_mode="aspect_cover_v2",
    ),
    PipelineVariant(
        key="task_aligned_5_7",
        label="当前最佳 5/7 + 强制对齐",
        use_rerank=True,
        rerank_mode="aspect_aware_conservative",
        answer_prompt_style="task_aligned",
        multi_snippet_count=1,
        final_source_count=5,
        complex_source_count=7,
        selection_mode="aspect_cover_v2",
    ),
)

SUMMARY_FILE_NAMES = {
    "base_router_4_6": "参数搜索_当前最佳4_6加按题作答_汇总.json",
    "task_aligned_4_6": "参数搜索_当前最佳4_6加强制对齐_汇总.json",
    "router_5_7": "参数搜索_当前最佳5_7加按题作答_汇总.json",
    "task_aligned_5_7": "参数搜索_当前最佳5_7加强制对齐_汇总.json",
}


def run_variant_parallel(
    pipeline: RagExperimentPipeline,
    prepared,
    variant: PipelineVariant,
    *,
    workers: int,
) -> list[Any]:
    if workers <= 1:
        results = []
        total = len(prepared.cases)
        for index, case in enumerate(prepared.cases, start=1):
            if index == 1 or index % 10 == 0 or index == total:
                print(f"[param-sweep] {variant.key}: {index}/{total}", flush=True)
            results.append(pipeline.run_case(prepared, case, variant))
        return results

    total = len(prepared.cases)
    ordered_results: list[Any] = [None] * total
    completed = 0
    with ThreadPoolExecutor(max_workers=workers) as executor:
        future_to_index = {
            executor.submit(pipeline.run_case, prepared, case, variant): index
            for index, case in enumerate(prepared.cases)
        }
        for future in as_completed(future_to_index):
            index = future_to_index[future]
            ordered_results[index] = future.result()
            completed += 1
            if completed == 1 or completed % 10 == 0 or completed == total:
                print(f"[param-sweep] {variant.key}: {completed}/{total}", flush=True)
    return ordered_results


def main() -> None:
    parser = argparse.ArgumentParser(description="运行最终参数配置对比搜索。")
    parser.add_argument("--embedding-model", default=DEFAULT_EMBEDDING_MODEL)
    parser.add_argument("--llm-model", default=DEFAULT_LLM_MODEL)
    parser.add_argument("--qa-2doc-samples", type=int, default=20)
    parser.add_argument("--qa-3doc-samples", type=int, default=20)
    parser.add_argument("--distractor-count", type=int, default=600)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=3)
    parser.add_argument("--output-root", default="", help="可选输出目录；为空时写入 results/02_最终参数搜索_20260517/")
    args = parser.parse_args()

    output_root = Path(args.output_root).resolve() if args.output_root else OUTPUT_ROOT
    ensure_dir(output_root)
    cache_root = ensure_dir(ROOT / ".cache")

    cases, docs = load_crud_cases(
        summary_samples=0,
        qa_1doc_samples=0,
        qa_2doc_samples=args.qa_2doc_samples,
        qa_3doc_samples=args.qa_3doc_samples,
        hallu_samples=0,
        negative_samples=0,
        distractor_count=args.distractor_count,
        seed=args.seed,
    )
    expected_count = args.qa_2doc_samples + args.qa_3doc_samples
    if len(cases) != expected_count:
        raise RuntimeError(f"CRUD 子样本当前评测批次数异常，期望 {expected_count}，实际 {len(cases)}")

    pipeline = RagExperimentPipeline(
        cache_root=cache_root,
        embedding_model=args.embedding_model,
        llm_model=args.llm_model,
    )
    prepared = pipeline.prepare_dataset(
        "crud_param_sweep",
        cases,
        docs,
        include_contextual=False,
        include_parent_child=False,
        include_query_rewrite=False,
    )

    rows: list[dict[str, Any]] = []
    summaries: list[dict[str, Any]] = []
    for variant in VARIANTS:
        results = run_variant_parallel(pipeline, prepared, variant, workers=max(1, args.workers))
        evaluation = evaluate_crud_results(
            variant.key,
            results,
            cases,
            ragas_case_ids=(),
            qa_ragas_case_ids=(),
            multidoc_ragas_case_ids=(),
            enable_ragas=False,
            semantic_model_name=args.embedding_model,
        )
        summary = dict(evaluation.summary)
        summary["label"] = variant.label
        summaries.append(summary)
        (output_root / SUMMARY_FILE_NAMES[variant.key]).write_text(
            json.dumps(summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        rows.append(
            {
                "variant": variant.key,
                "label": variant.label,
                "hit1": summary["retrieval_hit_rate_at_1"],
                "hit3": summary["retrieval_hit_rate_at_3"],
                "integration_string_similarity": summary["integration_string_similarity"],
                "integration_focus_f1": summary["integration_focus_f1"],
                "integration_quality": summary["integration_quality"],
                "complex_quality": summary["complex_quality"],
                "p50": summary["latency_p50_ms"],
            }
        )

    manifest = {
        "crud_subset_total": sum(CRUD_SUBSET_SPLITS.values()),
        "crud_subset_splits": CRUD_SUBSET_SPLITS,
        "evaluation_batch_size": len(cases),
        "evaluation_batch_splits": {
            "questanswer_2docs": args.qa_2doc_samples,
            "questanswer_3docs": args.qa_3doc_samples,
        },
        "embedding_model": args.embedding_model,
        "llm_model": args.llm_model,
        "distractor_count": args.distractor_count,
        "seed": args.seed,
        "workers": args.workers,
        "case_ids": [case.case_id for case in cases],
        "variants": [variant.label for variant in VARIANTS],
    }
    (output_root / "参数搜索_路线总表.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_root / "参数搜索_全部汇总.json").write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_root / "评测批次说明.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"output_root": str(output_root), "rows": rows}, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 评测批次说明

- 文件：`../results/02_最终参数搜索_20260517/评测批次说明.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/02_最终参数搜索_20260517/评测批次说明.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "dataset": "crud_param_search_batch",
  "crud_subset_total": 1594,
  "crud_subset_splits": {
    "questanswer_2docs": 797,
    "questanswer_3docs": 797
  },
  "evaluation_batch_size": 40,
  "evaluation_batch_splits": {
    "questanswer_2docs": 20,
    "questanswer_3docs": 20
  },
  "embedding_model": "bge-m3:latest",
  "llm_model": "qwen2.5:7b-instruct",
  "distractor_count": 600,
  "seed": 42,
  "workers": 3,
  "case_ids": [
    "questanswer_2docs_001",
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_004",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_007",
    "questanswer_2docs_008",
    "questanswer_2docs_009",
    "questanswer_2docs_010",
    "questanswer_2docs_011",
    "questanswer_2docs_012",
    "questanswer_2docs_013",
    "questanswer_2docs_014",
    "questanswer_2docs_015",
    "questanswer_2docs_016",
    "questanswer_2docs_017",
    "questanswer_2docs_018",
    "questanswer_2docs_019",
    "questanswe

### 参数搜索路线总表

- 文件：`../results/02_最终参数搜索_20260517/参数搜索_路线总表.json`

In [2]:
from pathlib import Path
import json

path = Path('../results/02_最终参数搜索_20260517/参数搜索_路线总表.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


[
  {
    "variant": "base_router_4_6",
    "label": "当前最佳 4/6 + 路由",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.5219,
    "integration_focus_f1": 0.6406,
    "integration_quality": 0.7032,
    "complex_quality": 0.7032,
    "p50": 4017.88
  },
  {
    "variant": "task_aligned_4_6",
    "label": "当前最佳 4/6 + 强制对齐",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.5175,
    "integration_focus_f1": 0.6156,
    "integration_quality": 0.6936,
    "complex_quality": 0.6936,
    "p50": 2510.53
  },
  {
    "variant": "router_5_7",
    "label": "当前最佳 5/7 + 路由",
    "hit1": 0.9,
    "hit3": 1.0,
    "integration_string_similarity": 0.5282,
    "integration_focus_f1": 0.6034,
    "integration_quality": 0.6896,
    "complex_quality": 0.6896,
    "p50": 4447.68
  },
  {
    "variant": "task_aligned_5_7",
    "label": "当前最佳 5/7 + 强制对齐",
    "hit1": 0.9,
    "hit3": 1.0,
    "integration_string_similarity": 0.5218,
    "integration_focus_f1":

### 参数搜索全部汇总

- 文件：`../results/02_最终参数搜索_20260517/参数搜索_全部汇总.json`

In [3]:
from pathlib import Path
import json

path = Path('../results/02_最终参数搜索_20260517/参数搜索_全部汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


[
  {
    "variant": "task_aligned_4_6",
    "dataset": "crud",
    "faithfulness": 0.0,
    "answer_correctness": 0.0,
    "answer_relevancy": 0.0,
    "context_precision": 0.0,
    "ragas_sample_count": 0,
    "accuracy": 0.0,
    "qa_accuracy": 0.0,
    "retrieval_hit_rate_at_1": 0.9,
    "retrieval_hit_rate_at_3": 0.975,
    "qa_faithfulness": 0.0,
    "qa_answer_correctness": 0.0,
    "qa_answer_relevancy": 0.0,
    "qa_context_precision": 0.0,
    "qa_ragas_sample_count": 0,
    "qa_similarity": 0.8842,
    "qa_string_similarity": 0.5175,
    "multidoc_faithfulness": 0.0,
    "multidoc_answer_correctness": 0.0,
    "multidoc_answer_relevancy": 0.0,
    "multidoc_context_precision": 0.0,
    "multidoc_ragas_sample_count": 0,
    "overall_similarity": 0.8842,
    "overall_string_similarity": 0.5175,
    "summary_similarity": 0.0,
    "summary_string_similarity": 0.0,
    "noise_robustness": 0.0,
    "negative_rejection": 0.0,
    "information_integration": 0.0,
    "integration_sim